# Multiparametric hydrogen production

Solve a family of hydrogen-production LPs with uncertain demand and water availability. Energia builds the electricity and material balances. Gana registers the two right-hand-side parameters and calls PPOPT to compute regions of the parameter space where the optimal dispatch follows an affine rule.

This compact port retains hydrogen demand and water availability as parameters. Wind and PV limits are fixed for the scenario, and storage is omitted from this single-period study.

## Formulate the LP

In [1]:
from math import isclose

import pandas as pd
from energia import Model, Process, Resource
from gana import T, inf

m = Model("parametric_hydrogen")
m.declare(Resource, ["sun", "wind", "power", "water", "hydrogen", "oxygen"])
_ = m.sun.consume <= 100
_ = m.wind.consume <= 100
water = m.water.consume
_ = water >= 0
delivered = m.hydrogen.release
_ = delivered >= 0
_ = m.oxygen.release >= 0

m.pv = Process(m.power == -(1 / 0.75) * m.sun)
m.wind_farm = Process(m.power == -(1 / 0.85) * m.wind)
m.electrolyzer = Process(
    m.hydrogen == -(1 / 0.019) * m.power - (0.1753 / 0.019) * m.water
    + (0.7632 / 0.019) * m.oxygen
)
for plant in (m.pv, m.wind_farm, m.electrolyzer):
    _ = plant.operate >= 0
    plant.locate()

m.program.demand = T(_=(0.5, 2.5))
m.program.water_supply = T(_=(6, 28))
m.program.meet_demand = delivered.V()[0] >= m.program.demand[0]
m.program.water_limit = water.V()[0] <= m.program.water_supply[0]
m.program.cost = inf(567 * m.pv.operate.V()[0] + 990 * m.wind_farm.operate.V()[0] + 0.001 * water.V()[0])
m.solve()
assert m.program.solution.critical_regions

## Evaluate two parameter realizations

Solar generation is cheaper in this model. Larger hydrogen requests use wind once solar potential is exhausted. Each evaluated point must also satisfy its water limit.

In [2]:
rows = []
for request, available in [(0.75, 9.0), (2.0, 22.0)]:
    dispatch = m.eval(request, available)
    production = dispatch[delivered.V()[0]]
    water_used = dispatch[water.V()[0]]
    assert isclose(production, request, abs_tol=1e-4)
    assert water_used <= available + 1e-4
    rows.append({
        "Hydrogen requested": request,
        "Water available": available,
        "Water used": water_used,
        "Solar generation": dispatch[m.pv.operate.V()[0]],
        "Wind generation": dispatch[m.wind_farm.operate.V()[0]],
    })

result = pd.DataFrame(rows)
assert result.iloc[0]["Wind generation"] < result.iloc[1]["Wind generation"]
result.round(3)

,Hydrogen requested,Water available,Water used,Solar generation,Wind generation
0,0.75,9.0,6.920,39.474,-0.000
1,2.00,22.0,18.453,75.000,30.263
